## 0️⃣ Clone Repository (Run First)

**Run this cell first** to clone the repo from GitHub using your Personal Access Token stored in Kaggle Secrets.

In [ ]:
# Clone repository using GITHUB_TOKEN from Kaggle Secrets
import os
import sys

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")
REPO_URL = "https://github.com/<YOUR_USERNAME>/clips-kitty-kaggle.git"  # ← REPLACE with your repo
CLONE_DIR = "/tmp/clips-kitty-kaggle"

if not GITHUB_TOKEN:
    print("❌ GITHUB_TOKEN not found in environment.")
    print("   1. Create a PAT at github.com/settings/tokens (scope: repo)")
    print("   2. Add to Kaggle Secrets (🔑) as GITHUB_TOKEN")
    print("   3. Replace <YOUR_USERNAME> in REPO_URL above")
else:
    import subprocess
    clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
    result = subprocess.run(["git", "clone", clone_url, CLONE_DIR], capture_output=True, text=True)
    if result.returncode == 0:
        print(f"✅ Cloned to {CLONE_DIR}")
    else:
        print(f"❌ Clone failed: {result.stderr}")

sys.path.insert(0, CLONE_DIR)
%cd {CLONE_DIR}

# Clips Kitty on Kaggle 🎯

Extract viral clips from YouTube, Twitch, and Kick videos using AI-powered content analysis.

**Original Project:** [Clips Kitty](https://github.com/Clips-Kitty/Clips-Kitty) — Windows-first Electron + Python desktop app

**This Notebook:** Kaggle-adapted version using the original pipeline via adapter layer — zero modifications to core processing logic.

---

## Quick Start

1. **Cell 1** — Clone repo (run first, needs GITHUB_TOKEN in Secrets)
2. **Cells 4-5** — Install dependencies, verify imports
3. **Cells 7-8** — Detect GPU, configure environment
4. **Cell 12** — Paste YouTube URL or upload video
5. **Cell 14** — Select content mode, clip count, duration
6. **Cells 16-17** — Run pipeline with live progress
7. **Cells 19, 21, 23** — Preview clips, download individually or as ZIP

---

## Requirements
- Kaggle GPU notebook (T4, P100, V100, A100, or L4)
- Internet access for model downloads (first run only)
- **Optional:** Add `OPENROUTER_API_KEY` to Kaggle Secrets for cloud LLM reranking
- **Required:** Add `GITHUB_TOKEN` to Kaggle Secrets for private repo access

## 1️⃣ Environment Setup & Dependency Installation

In [ ]:
# Install dependencies (skip PyTorch — use Kaggle's pre-installed version)
!pip install -r requirements-kaggle.txt -q 2>&1 | tail -30

In [ ]:
# Comprehensive environment verification
import sys
import torch
import torchvision
import cv2
import faster_whisper
import ultralytics
import yt_dlp
import huggingface_hub
import onnxruntime
import rapidocr_onnxruntime

print("=" * 60)
print("KAGGLE ENVIRONMENT VERIFICATION")
print("=" * 60)

print(f"\nPython: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"torchvision: {torchvision.__version__}")
print(f"CUDA (PyTorch): {torch.version.cuda}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"CUDA Capability: {torch.cuda.get_device_capability(0)}")

print(f"\nOpenCV: {cv2.__version__}")
print(f"faster-whisper: {faster_whisper.__version__}")
print(f"ultralytics: {ultralytics.__version__}")
print(f"yt-dlp: {yt_dlp.version.__version__}")
print(f"huggingface-hub: {huggingface_hub.__version__}")
print(f"onnxruntime: {onnxruntime.__version__}")
print(f"rapidocr-onnxruntime: {rapidocr_onnxruntime.__version__}")

# Check CTranslate2 (used by faster-whisper)
try:
    import ctranslate2
    print(f"CTranslate2: {ctranslate2.__version__}")
    print(f"CTranslate2 CUDA support: {ctranslate2.get_cuda_version()}")
except ImportError:
    print("CTranslate2: NOT INSTALLED")

print("\n" + "=" * 60)
print("GPU COMPATIBILITY TEST: CTranslate2 / faster-whisper")
print("=" * 60)

# Test CTranslate2 GPU support - lightweight, no model loading
try:
    import ctranslate2
    import torch
    cuda_version = ctranslate2.get_cuda_version()
    if cuda_version:
        print(f"✅ CTranslate2 reports CUDA: {cuda_version}")
        # Quick GPU test without loading models
        test_tensor = torch.zeros(1, device='cuda')
        del test_tensor
        torch.cuda.empty_cache()
        print("✅ CUDA device accessible")
    else:
        print("⚠️ CTranslate2 reports NO CUDA support — will fall back to CPU")
except Exception as e:
    print(f"❌ CTranslate2/GPU test failed: {e}")
    print("   Will use CPU for transcription")

print("=" * 60)

## 2️⃣ GPU Detection & Configuration

In [ ]:
# Detect Kaggle environment and hardware
from kaggle_adapter.environment import detect_kaggle_environment, print_environment_summary

env = detect_kaggle_environment()
print_environment_summary(env)

In [ ]:
# Quick setup — configures everything for Kaggle
from kaggle_adapter.config import quick_setup

# Adjust these settings as needed:
config = quick_setup(
    data_dir="/data",           # Persistent volume (survives kernel restarts)
    use_cloud_llm=True,          # Requires OPENROUTER_API_KEY in Kaggle Secrets
    force_cpu=False,             # Set True to force CPU mode
    verbose=True
)

# Verify config loaded
print(f"\n✅ Config ready")
print(f"   Data dir: {config.settings['paths']['data_dir']}")
print(f"   Device: {config.settings['transcription']['device']}")
print(f"   Model: {config.settings['transcription']['model']}")
print(f"   Parallel renders: {config.settings['video']['parallel_renders']}")
print(f"   Encoder: {config.settings['video']['encoder']}")
print(f"   LLM Backend: {config.settings['llm']['backend']}")

## 3️⃣ Original Clips Kitty Repository Setup

In [ ]:
# Verify original pipeline imports
from core.pipeline import process_video
from core.state import StateDB
from core.models import ClipCandidate, RenderedClip, Segment
from sources.dispatch import identify

print("✅ Original Clips Kitty pipeline imports successful")
print(f"   process_video: {process_video}")
print(f"   StateDB: {StateDB}")
print(f"   ClipCandidate: {ClipCandidate}")
print(f"   RenderedClip: {RenderedClip}")
print(f"   identify: {identify}")

## 4️⃣ Video Input

In [ ]:
# Choose input method:

### Option A: YouTube URL
# VIDEO_URL = "https://youtu.be/dQw4w9WgXcQ"  # Replace with your URL

### Option B: Local file from /kaggle/input/ (update path as needed)
VIDEO_URL = "/kaggle/input/datasets/chaitanyashimpi33/clip-1/joy-koy.mp4"

### Option C: Local file already in /data/downloads/ (use "local:" prefix)
# VIDEO_URL = "local:my_video"

# Detect source type and route to correct adapter method
from sources.dispatch import identify
from pathlib import Path

source_type, video_id = identify(VIDEO_URL)
print(f"Source: {source_type}")
print(f"Video ID: {video_id}")
print(f"URL: {VIDEO_URL}")

# Determine which adapter method to use
is_local_file = source_type == "local" or (not VIDEO_URL.startswith(("http", "https", "local:")) and Path(VIDEO_URL).exists())

if is_local_file:
    if source_type != "local":
        # Raw file path — extract video_id from filename
        video_id = Path(VIDEO_URL).stem
    print(f"→ Will use adapter.process_local_file() for: {VIDEO_URL}")
else:
    print(f"→ Will use adapter.process_url() for: {VIDEO_URL}")

## 5️⃣ Processing Configuration

In [ ]:
# Content Mode Selection
print("Available Content Modes:")
print("  1. Standard (Shorts)      — General purpose, vertical crop with speaker tracking")
print("  2. Vertical Live          — Full frame vertical, no crop (for vertical source videos)")
print("  3. Sports                 — Play-aware boundaries, scoreboard detection")
print("  4. Gaming                 — Webcam layout detection, game audio peaks")
print("  5. Podcast                — Multi-speaker static crops, shot-aware")
print("  6. Longform               — 16:9 horizontal, chapter markers")

CONTENT_MODE = "standard"  # Options: standard, vertical_live, sports, gaming, podcast, longform

# Clip Settings
CLIP_COUNT = 3              # Number of clips to generate (1-10)
MIN_DURATION = 15           # Minimum clip duration in seconds
MAX_DURATION = 60           # Maximum clip duration in seconds (Shorts limit)
MIN_SCORE = 0.6             # Minimum fusion score threshold (0.0-1.0)

# Advanced Settings (affect clip quality)
print("\nAdvanced Settings (modify if needed):")
print(f"  Transcription model: {config.settings['transcription']['model']} (larger = more accurate, slower)")
print(f"  Vertical crop mode: {config.settings['clips']['vertical']['crop_mode']} (track/split/fit_blur)")
print(f"  Captions enabled: {config.settings['clips']['captions']['enabled']}")
print(f"  Outro enabled: {config.settings['video']['outro']['enabled']}")
print(f"  Parallel renders: {config.settings['video']['parallel_renders']} (increase for A100 80GB)")

# Override config with our settings
config.settings["clips"]["vertical"]["max_duration"] = MAX_DURATION
config.settings["clips"]["vertical"]["min_duration"] = MIN_DURATION
config.settings["clips"]["scoring"]["min_score"] = MIN_SCORE

# Content mode specific adjustments
if CONTENT_MODE == "vertical_live":
    config.settings["clips"]["vertical"]["crop_mode"] = "fit_blur"
elif CONTENT_MODE == "sports":
    config.settings["clips"]["vertical"]["crop_mode"] = "track"
elif CONTENT_MODE == "longform":
    config.settings["clips"]["vertical"]["enabled"] = False
    config.settings["video"]["encoder"] = "libx264"  # Better quality for longform

print(f"\n✅ Configuration applied for {CONTENT_MODE} mode")

## 6️⃣ Video Processing

In [ ]:
# Initialize pipeline adapter with progress monitoring
from kaggle_adapter.pipeline_adapter import PipelineAdapter, PipelineResult
from kaggle_adapter.progress_monitor import ProgressMonitor, create_progress_callback

# Create progress monitor (rich UI for notebooks)
monitor = ProgressMonitor(
    use_tqdm=True,
    use_ipython=True,
    refresh_rate=0.5
)
monitor.start()

# Create adapter with progress callback
adapter = PipelineAdapter(
    config=config,
    data_dir="/data",
    use_cloud_llm=True,
    force_cpu=False,
    progress_callback=create_progress_callback(monitor),
    verbose=True
)

print("🚀 Starting pipeline...")
print(f"   URL: {VIDEO_URL}")
print(f"   Mode: {CONTENT_MODE}")
print(f"   Target clips: {CLIP_COUNT}")
print(f"   Duration: {MIN_DURATION}-{MAX_DURATION}s")
print("   \nProgress will appear below...")

In [ ]:
# Run the pipeline
try:
    if is_local_file:
        result = adapter.process_local_file(VIDEO_URL, force=False)
    else:
        result = adapter.process_url(VIDEO_URL, force=False)
    
    # Finish monitoring
    monitor.finish(result.success, result.error or "")
    
    if result.success:
        print(f"\n✅ Pipeline completed successfully!")
        print(f"   Video ID: {result.video_id}")
        print(f"   Title: {result.video_title}")
        print(f"   Clips created: {len(result.clips)}")
        print(f"   Processing time: {result.processing_time_seconds:.1f}s")
        print(f"   Stages: {', '.join(result.stages_completed)}")
        
        # Show clip details
        for i, clip in enumerate(result.clips):
            c = clip.candidate
            print(f"\n   Clip {i+1}: {c.start:.1f}s - {c.end:.1f}s ({c.duration:.1f}s)")
            print(f"      Score: {c.score} | Hook: {c.hook[:60]}...")
            print(f"      Path: {clip.path}")
    else:
        print(f"\n❌ Pipeline failed: {result.error}")
        print(f"   Stages completed: {', '.join(result.stages_completed)}")
        if result.error_traceback:
            print(f"\n--- FULL TRACEBACK ---")
            print(result.error_traceback)
        
except KeyboardInterrupt:
    print("\n⚠️ Pipeline interrupted by user")
    adapter.cancel()
    monitor.finish(False, "Interrupted")
except Exception as e:
    print(f"\n❌ Unexpected error: {e}")
    import traceback
    traceback.print_exc()
    monitor.finish(False, str(e))
finally:
    adapter.close()

## 7️⃣ Generated Clip Previews

In [ ]:
# Preview clips in notebook
if result.success and result.clips:
    from IPython.display import Video, display
    
    print(f"Previewing {len(result.clips)} clips:\n")
    
    for i, clip in enumerate(result.clips):
        c = clip.candidate
        print(f"### Clip {i+1} — {c.start:.1f}s to {c.end:.1f}s ({c.duration:.1f}s) — Score: {c.score}")
        print(f"Hook: {c.hook}")
        print(f"Reason: {c.reason}")
        
        # Display video player
        try:
            display(Video(str(clip.path), width=480, html_attributes="controls"))
        except Exception as e:
            print(f"   ⚠️ Could not preview: {e}")
        print("---")
else:
    print("No clips to preview. Run the pipeline first.")

## 8️⃣ Download Individual Clips

In [ ]:
# Export clips to Kaggle working directory for download
if result.success and result.clips:
    print("Exporting clips to /kaggle/working/clips/...")
    
    exported_paths = adapter.export_clips_to_working(result.video_id)
    
    print(f"✅ Exported {len(exported_paths)} clips:")
    for path in exported_paths:
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"   {path.name} ({size_mb:.1f} MB)")
    
    print(f"\n📁 Clips saved to: /kaggle/working/clips/{result.video_id}/")
    print("   Use the file browser (left sidebar) to download individual clips.")
else:
    print("No clips to export. Run the pipeline first.")

## 🔐 Download All Clips as ZIP

In [ ]:
# Create ZIP archive of all clips
if result.success and result.clips:
    import zipfile
    from pathlib import Path
    
    zip_path = Path(f"/kaggle/working/clips/{result.video_id}.zip")
    
    print(f"Creating ZIP: {zip_path.name}")
    
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for clip_path in adapter.get_clip_paths(result.video_id):
            zf.write(clip_path, arcname=clip_path.name)
    
    zip_size_mb = zip_path.stat().st_size / (1024 * 1024)
    print(f"✅ ZIP created: {zip_path.name} ({zip_size_mb:.1f} MB)")
    print(f"   Download from file browser: /kaggle/working/clips/{zip_path.name}")
else:
    print("No clips to zip. Run the pipeline first.")

## 📋 Session Summary & Re-run

In [ ]:
# Get processing summary
if 'result' in locals() and result.success:
    summary = monitor.get_summary()
    
    print("=" * 50)
    print("PROCESSING SUMMARY")
    print("=" * 50)
    print(f"Video ID:      {summary['video_id']}")
    print(f"Total Time:    {summary['total_time']:.1f}s")
    print(f"Clips Created: {len(result.clips)}")
    print("\nStage Breakdown:")
    for stage_name, stage_info in summary['stages'].items():
        status = "✅" if stage_info['completed'] else "⏸️"
        dur = stage_info['duration']
        print(f"  {status} {stage_name:12s} — {dur:.1f}s — {stage_info['message']}")
    
    print("\n" + "=" * 50)
    print("To re-run with different settings:")
    print("1. Modify CONTENT_MODE, CLIP_COUNT, MIN_DURATION, MAX_DURATION in Cell 14")
    print("2. Re-run cells 14, 16, 17, 19, 21, 23")
    print("3. The pipeline will skip already-processed stages (cached in /data/state.db)")

## 🔧 Troubleshooting & Tips

In [ ]:
# Common issues and solutions
print("""
COMMON ISSUES:
==============

1. CUDA OOM (Out of Memory):
   - Set force_cpu=True in quick_setup()
   - Reduce parallel_renders to 1
   - Use smaller Whisper model: config.settings['transcription']['model'] = 'small'

2. yt-dlp download fails:
   - Add YouTube cookies to Kaggle Secrets as YOUTUBE_COOKIES
   - Or place cookies.txt in /data/youtube_cookies.txt
   - Update yt-dlp: !pip install -U yt-dlp

3. No clips generated:
   - Lower MIN_SCORE (try 0.4-0.5)
   - Increase MAX_DURATION
   - Check video has speech (Whisper needs audio)

4. Slow processing:
   - Ensure GPU is detected (check Cell 2 output)
   - Use cloud LLM (add OPENROUTER_API_KEY to Kaggle Secrets)
   - Models auto-cache on first run

5. Interrupted session:
   - State is saved in /data/state.db — re-run will resume
   - Clips already rendered won't be re-rendered

USEFUL COMMANDS:
================

# Check disk usage
!du -sh /data/*

# Clear cache and restart
!rm -rf /data/state.db /data/downloads/* /data/transcripts/*

# View pipeline logs
!cat /data/state.db  # (SQLite, use DB browser)
""")

## 🔐 Required Kaggle Secrets

Add these to Kaggle Secrets (🔑 icon in left sidebar):

| Secret Name | Required? | Purpose |
|-------------|-----------|---------|
| `GITHUB_TOKEN` | **Yes** (if repo private) | GitHub PAT with `repo` scope |
| `OPENROUTER_API_KEY` | Recommended | Cloud LLM for best clip scoring |
| `YOUTUBE_COOKIES` | Optional | Age-restricted/private video access |
| `OPENAI_API_KEY` | Alternative | OpenAI models instead of OpenRouter |
| `ANTHROPIC_API_KEY` | Alternative | Claude models instead of OpenRouter |